# CoalGameRec — Round-4 Required Experiments

Runs the four experiment scripts prepared for the round-4 review response. Each cell streams output live **and** the scripts persist logs + CSV artifacts you can hand back.

**Recommended order (by reviewer priority):**

| Cell | Experiment | Script | Est. time (Apple Silicon) |
|---|---|---|---|
| 3 | Design ablations: k-sweep, player-selection, hard-vs-smooth utility, native-vs-external | `run_design_ablations.py` | ~4 h Amazon / ~10-12 h ML-1M |
| 4 | Shapley estimator convergence (M=16…256, efficiency residuals) | `run_estimator_convergence.py` | ~2-3 h ML-1M |
| 5 | TRUE masked-forward faithfulness (edge removal + re-propagation) | `run_masked_forward_faithfulness.py` | ~30-40 min / dataset |
| 6 | C1 re-run **with Shapley** (closes C1 scope gap) | `run_matched_controls.py` + `C1_WITH_SHAPLEY=1` | ~2.5 h Amazon / ~8 h ML-1M |

**Notes**
- Cells 3–5 retrain from scratch (not resume-safe); Cell 6 is resume-safe (completed seeds skip).
- Artifacts land in `results/journal_runs/<run>/tables/*.csv` + `<name>.log` next to them.
- Keep the notebook tab open while a cell runs. If the kernel dies mid-cell, just re-run that cell.
- Run Cell 1 first, then Cell 2 to confirm the environment.

In [ ]:
# Cell 1 — Setup: paths + device detection
from pathlib import Path
import os, sys, platform

CWD = Path.cwd().resolve()
CODE_DIR = CWD.parent if CWD.name == "notebooks" else CWD
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
RESULTS = CODE_DIR / "results" / "journal_runs"

import torch
if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

RUN_ENV = dict(os.environ, COALGAME_DEVICE=DEVICE)

print("CODE_DIR:", CODE_DIR)
print("device  :", DEVICE, "| torch", torch.__version__, "| python", platform.python_version())
print("platform:", platform.platform())

In [ ]:
# Cell 2 — Environment check (source runs + splits present?)
required = {
    "ML-1M v3 run": RESULTS / "ml1m_lightgcn_v3_prospective" / "splits" / "train.parquet",
    "Amazon v3 run": RESULTS / "amazon_books_lightgcn_v3_prospective" / "splits" / "train.parquet",
    "C1 Amazon run": RESULTS / "amazon_books_lightgcn_v4_matched_controls" / "tables" / "summary_mean_std.csv",
}
ok = True
for name, path in required.items():
    print(f"{'OK ' if path.exists() else 'MISSING'} {name}: {path}")
    ok = ok and path.exists()
for script in ["run_design_ablations.py", "run_estimator_convergence.py",
               "run_masked_forward_faithfulness.py", "run_matched_controls.py"]:
    p = CODE_DIR / "scripts" / script
    print(f"{'OK ' if p.exists() else 'MISSING'} script: {p.name}")
    ok = ok and p.exists()
print("\nReady." if ok else "\nFix missing pieces before running experiments.")

In [ ]:
# Cell 3 — EXPERIMENT 1: design ablations (Critical/High: k-sweep, player selection,
# hard-vs-smooth utility, native-vs-external intervention). Set DATASETS as needed.
import subprocess

DATASETS = ["amazon", "ml1m"]   # reorder or drop one to control runtime
SEED = 42

for ds in DATASETS:
    cmd = [sys.executable, str(CODE_DIR / "scripts" / "run_design_ablations.py"),
           "--dataset", ds, "--seed", str(SEED)]
    print("=" * 70)
    print("RUN:", " ".join(cmd))
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          cwd=str(CODE_DIR), env=RUN_ENV, text=True, bufsize=1) as p:
        for line in p.stdout:
            print(line, end="")
    print(f"\n[{ds}] exit code:", p.returncode)
    assert p.returncode == 0, f"design ablations failed for {ds}"
print("\nDONE. Artifacts: results/journal_runs/<run>/tables/design_ablations.csv")

In [ ]:
# Cell 4 — EXPERIMENT 2: Shapley estimator convergence (M=16,32,64,128,256 x 2 RNG seeds).
# Reports runtime, efficiency residual, Spearman/l1/l2 vs the M=256 reference.
# Default subsamples 1000 users for feasibility; set MAX_USERS=None for all users (very long).
import subprocess

DATASET = "ml1m"      # or "amazon"
SEED = 42
MAX_USERS = 1000

cmd = [sys.executable, str(CODE_DIR / "scripts" / "run_estimator_convergence.py"),
       "--dataset", DATASET, "--seed", str(SEED)]
if MAX_USERS:
    cmd += ["--max-users", str(MAX_USERS)]
print("RUN:", " ".join(cmd))
with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      cwd=str(CODE_DIR), env=RUN_ENV, text=True, bufsize=1) as p:
    for line in p.stdout:
        print(line, end="")
print("\nexit code:", p.returncode)
print("DONE. Artifact: tables/estimator_convergence.csv")

In [ ]:
# Cell 5 — EXPERIMENT 3: TRUE masked-forward faithfulness.
# Removes/keeps top-attributed player edges, rebuilds normalized adjacency,
# re-propagates the frozen LightGCN, and evaluates test-item rank.
import subprocess

DATASETS = ["ml1m", "amazon"]
SEED = 42
N_USERS = 1000      # subsample size

for ds in DATASETS:
    cmd = [sys.executable, str(CODE_DIR / "scripts" / "run_masked_forward_faithfulness.py"),
           "--dataset", ds, "--seed", str(SEED), "--n-users", str(N_USERS)]
    print("=" * 70)
    print("RUN:", " ".join(cmd))
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          cwd=str(CODE_DIR), env=RUN_ENV, text=True, bufsize=1) as p:
        for line in p.stdout:
            print(line, end="")
    print(f"\n[{ds}] exit code:", p.returncode)
    assert p.returncode == 0, f"masked-forward faithfulness failed for {ds}"
print("\nDONE. Artifacts: tables/masked_forward_faithfulness.csv")

In [ ]:
# Cell 6 — EXPERIMENT 4: C1 re-run WITH SHAPLEY (closes the C1 scope gap).
# Resume-safe: completed seeds are skipped. Amazon first (fast); ML-1M optional (long).
import subprocess

RUNS = [
    # (dataset, source v4 run, output v4b run)
    ("amazon", "amazon_books_lightgcn_v4_matched_controls", "amazon_books_lightgcn_v4b_matched_controls"),
    # ("ml1m", "ml1m_lightgcn_v4_matched_controls", "ml1m_lightgcn_v4b_matched_controls"),  # ~8 h — uncomment when ready
]
SEEDS = [42, 43, 44, 45, 46]

env = dict(RUN_ENV, C1_WITH_SHAPLEY="1")
for ds, src, out in RUNS:
    cmd = [sys.executable, str(CODE_DIR / "scripts" / "run_matched_controls.py"),
           "--dataset", ds,
           "--source-run", str(RESULTS / src),
           "--out", str(RESULTS / out),
           "--seeds", *[str(s) for s in SEEDS]]
    print("=" * 70)
    print("RUN:", " ".join(cmd))
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          cwd=str(CODE_DIR), env=env, text=True, bufsize=1) as p:
        for line in p.stdout:
            print(line, end="")
    print(f"\n[{ds}] exit code:", p.returncode)
    assert p.returncode == 0, f"C1-with-Shapley failed for {ds}"
print("\nDONE. New runs contain the shapley-mc family alongside all C1 families.")

In [ ]:
# Cell 7 — Status & artifacts overview (safe to run any time)
from pathlib import Path

print("=== experiment artifacts ===")
checks = [
    ("design_ablations.csv (ml1m)", RESULTS / "ml1m_lightgcn_v3_prospective" / "tables" / "design_ablations.csv"),
    ("design_ablations.csv (amazon)", RESULTS / "amazon_books_lightgcn_v3_prospective" / "tables" / "design_ablations.csv"),
    ("estimator_convergence.csv", RESULTS / "ml1m_lightgcn_v3_prospective" / "tables" / "estimator_convergence.csv"),
    ("masked_forward_faithfulness.csv (ml1m)", RESULTS / "ml1m_lightgcn_v3_prospective" / "tables" / "masked_forward_faithfulness.csv"),
    ("masked_forward_faithfulness.csv (amazon)", RESULTS / "amazon_books_lightgcn_v3_prospective" / "tables" / "masked_forward_faithfulness.csv"),
    ("C1-with-Shapley (amazon)", RESULTS / "amazon_books_lightgcn_v4b_matched_controls" / "tables" / "summary_mean_std.csv"),
    ("C1-with-Shapley (ml1m)", RESULTS / "ml1m_lightgcn_v4b_matched_controls" / "tables" / "summary_mean_std.csv"),
]
for name, path in checks:
    print(f"{'DONE ' if path.exists() else 'pending'} {name}")

print("\n=== recent log tails ===")
for log in sorted(RESULTS.glob("*/design_ablations.log")) + sorted(RESULTS.glob("*/estimator_convergence.log")) + sorted(RESULTS.glob("*/masked_forward_faithfulness.log")):
    tail = log.read_text(errors='ignore').splitlines()[-3:]
    print(f"\n{log.parent.parent.name}/{log.name}:")
    for t in tail:
        print("  ", t)

In [ ]:
# Cell 8 — Hand back to the loop: commit + push the artifacts
print("Run these from the REPO ROOT (next-paper/):")
print("""
git add paper-ideas/CoalGameRec/code/results/journal_runs paper-ideas/CoalGameRec/code/notebooks/CoalGameRec_R4_Experiments.ipynb
git commit -m "round-4 experiments: design ablations, M-convergence, masked-forward faithfulness, C1-with-Shapley"
git push origin arena/019fdd75-next-paper
""")